# Exploration

Working notebook. Keep it in the repo -- it shows process.

Rule of thumb: anything you would run twice belongs in `xtpress/`, not here.


In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from xtpress import config, loaders, expected_threat as xt

pd.set_option("display.max_columns", 50)
config.PROJECT_ROOT


WindowsPath('C:/Users/felip/Documents/xT')

## 1. First look at one match

Browse a single match before writing any parsing code. Check: what does
`pass_outcome` look like on a completed pass? How many values does
`shot_end_location` have?


In [3]:
events = loaders.load_events(3857276)   # WC22 Canada v Morocco
events.type.value_counts().head(15)


type
Pass              955
Ball Receipt*     868
Carry             782
Pressure          289
Ball Recovery      93
Duel               63
Clearance          41
Dribble            39
Miscontrol         32
Foul Committed     31
Foul Won           31
Block              27
Dispossessed       23
Dribbled Past      23
Interception       23
Name: count, dtype: int64

## 2. Build the action frame


In [4]:
actions = loaders.to_actions(events)
actions.head()


,competition,match_id,index,period,minute,second,time_seconds,possession,possession_team,team,team_id,player,player_id,position,action_type,start_x,start_y,end_x,end_y,successful,is_goal,xg,is_set_piece,is_penalty,under_pressure,counterpress,play_pattern,body_part
0,None,3857276,5,1,0,0,0.460,2,Morocco,Morocco,788,Youssef En-Nesyri,6301.0,Center Forward,pass,61.0,40.1,48.5,41.3,True,False,NaN,True,False,False,False,From Kick Off,Left Foot
1,None,3857276,7,1,0,1,1.295,2,Morocco,Morocco,788,Sofyan Amrabat,5234.0,Center Defensive Midfield,carry,48.5,41.3,48.4,42.5,True,False,NaN,False,False,False,False,From Kick Off,NaN
2,None,3857276,8,1,0,2,2.415,2,Morocco,Morocco,788,Sofyan Amrabat,5234.0,Center Defensive Midfield,pass,48.4,42.5,30.5,59.0,True,False,NaN,False,False,False,False,From Kick Off,Right Foot
3,None,3857276,10,1,0,3,3.886,2,Morocco,Morocco,788,Nayef Aguerd,12149.0,Right Center Back,carry,30.5,59.0,30.9,59.0,True,False,NaN,False,False,False,False,From Kick Off,NaN
4,None,3857276,11,1,0,5,5.143,2,Morocco,Morocco,788,Nayef Aguerd,12149.0,Right Center Back,pass,30.9,59.0,29.8,26.9,True,False,NaN,False,False,False,False,From Kick Off,Left Foot


## 3. Fit and eyeball the surface


In [5]:
model = xt.ExpectedThreat().fit(actions)
model.grid().round(3)


array([[0.   , 0.003, 0.003, 0.003, 0.003, 0.003, 0.005, 0.005, 0.005,
        0.004, 0.002, 0.   ],
       [0.001, 0.004, 0.004, 0.005, 0.004, 0.005, 0.007, 0.005, 0.007,
        0.006, 0.001, 0.001],
       [0.003, 0.003, 0.004, 0.005, 0.005, 0.005, 0.006, 0.006, 0.005,
        0.025, 0.049, 0.097],
       [0.005, 0.003, 0.006, 0.005, 0.005, 0.006, 0.011, 0.01 , 0.009,
        0.009, 0.097, 0.183],
       [0.003, 0.004, 0.004, 0.005, 0.006, 0.007, 0.008, 0.011, 0.012,
        0.023, 0.07 , 0.319],
       [0.004, 0.004, 0.004, 0.006, 0.007, 0.006, 0.007, 0.011, 0.015,
        0.02 , 0.043, 0.   ],
       [0.002, 0.003, 0.003, 0.005, 0.006, 0.008, 0.009, 0.011, 0.01 ,
        0.026, 0.037, 0.017],
       [0.002, 0.003, 0.003, 0.004, 0.006, 0.009, 0.012, 0.014, 0.016,
        0.009, 0.017, 0.095]])

## 4. Possession frame


In [8]:
from xtpress import loaders

ev = loaders.load_events(3857276)
pr = ev[ev["type"] == "Pressure"]

links = pr[["id", "team_id", "location", "related_events"]].explode("related_events")
links = links.dropna(subset=["related_events"])

merged = links.merge(
    ev[["id", "team_id", "location"]],
    left_on="related_events", right_on="id", suffixes=("_press", "_action"))
merged = merged[merged["team_id_press"] != merged["team_id_action"]]

p = np.vstack(merged["location_press"].map(lambda v: np.asarray(v, dtype=float)[:2]).to_numpy())
a = np.vstack(merged["location_action"].map(lambda v: np.asarray(v, dtype=float)[:2]).to_numpy())

shared = np.hypot(*(p - a).T)
mirrored = np.hypot(120 - p[:, 0] - a[:, 0], 80 - p[:, 1] - a[:, 1])
print(len(merged), np.median(shared), np.median(mirrored))

406 73.57427278347281 3.250380133568763
